# Section cohesion across 10 skill repositories

This notebook presents the corpus-scale output of `scripts/skill_corpus_section_cohesion.py`. The committed run covers 697 `SKILL.md` files from 10 repositories with complete cached coverage. The repositories are an exploratory convenience set, not a random sample of the full corpus, so conclusions apply to these repositories rather than all skill repositories.

In [ ]:
import csv
import json
import sys
from importlib import import_module
from pathlib import Path

from IPython.display import Image, display


def find_repo_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src/alexandria").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from the Alexandria checkout.")


REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT))
plot_repo_summary = import_module("scripts.skill_corpus_section_cohesion").plot_repo_summary

OUTPUT_DIR = REPO_ROOT / "notebooks/outputs/skill_corpus_section_cohesion"
with (OUTPUT_DIR / "skills.csv").open(encoding="utf-8") as input_file:
    skill_rows = list(csv.DictReader(input_file))
with (OUTPUT_DIR / "repos.csv").open(encoding="utf-8") as input_file:
    repo_rows = list(csv.DictReader(input_file))
global_test = json.loads((OUTPUT_DIR / "global_test.json").read_text(encoding="utf-8"))
print(f"loaded {len(skill_rows)} skills across {len(repo_rows)} repos")

## Measurement and comparison unit

Only optimizable body sentences are used. Each sentence belongs to its deepest Alexandria section. For every section containing at least two sentences, cohesion is the mean cosine similarity over all sentence pairs. Section scores are reduced to a median within each skill, and the graph compares those per-skill medians across repositories. This hierarchy prevents a long section or a repository with many sentences from dominating solely by size.

The global Kruskal–Wallis test asks whether at least one repository distribution differs. Each repository is also compared with all other selected repositories using a two-sided Mann–Whitney U test; colors use Benjamini–Hochberg-adjusted q-values. These tests establish distributional differences, not their cause.

In [ ]:
print(
    f"Kruskal-Wallis: H={global_test['statistic']:.3f}, p={global_test['p_value']:.3e}, "
    f"epsilon^2={global_test['epsilon_squared']:.3f}\n"
)
print("repo                                      skills  sections  median    95% bootstrap CI      q vs rest  delta")
for row in sorted(repo_rows, key=lambda item: float(item["median_skill_section_pairwise_cosine"]), reverse=True):
    print(
        f"{row['repo'][:41]:41} {int(row['analyzed_skill_count']):6d}  "
        f"{int(row['analyzed_section_count']):8d}  "
        f"{float(row['median_skill_section_pairwise_cosine']):.3f}  "
        f"[{float(row['bootstrap_ci_low']):.3f}, {float(row['bootstrap_ci_high']):.3f}]  "
        f"{float(row['vs_rest_bh_q_value']):9.2e}  {float(row['vs_rest_cliffs_delta']):+.3f}"
    )

## Repository distributions

Each dot is one skill's median section cohesion. The box shows the middle 50%, the black bar is the repository median, and the dashed vertical line is the median over all 697 analyzed skills. Orange and blue mean significantly higher or lower than the other selected repositories after multiple-testing correction; gray means the adjusted test did not provide evidence of a difference.

In [ ]:
graph_path = OUTPUT_DIR / "repos.png"
plot_repo_summary(skill_rows, repo_rows, global_test, graph_path)
display(Image(filename=str(graph_path)))

## Interpretation of this run

The repository distributions are not interchangeable (`p = 4.66e-40`), and the rank-based effect size is substantial (`epsilon² = 0.291`). `CloudAI-X__claude-workflow-v2`, `Owl-Listener__designer-skills`, and `Prat011__awesome-llm-skills` have the highest medians in this set (0.398, 0.383, and 0.378). `NVIDIA__skills` is lowest at 0.308; `K-Dense-AI__scientific-agent-skills` and `alirezarezvani__claude-skills` are around 0.327–0.328.

This is evidence that section cohesion varies materially by repository, but it does not show that section-local compression preserves document embeddings. Repository language, domain, templates, duplicated skills, parser behavior, and section size can all contribute. A compression-fidelity experiment still needs to test whether higher measured cohesion predicts smaller document-embedding drift after matched section-local compression.